<a href="https://colab.research.google.com/github/abdulafadil374-byte/Meta-ai-development-hackathon-/blob/main/Meta_ai_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio pandas requests langdetect lingua-language-detector

import os, re, json, time, html, tempfile, requests
import pandas as pd, gradio as gr
from datetime import datetime
from getpass import getpass
from langdetect import detect_langs, DetectorFactory
DetectorFactory.seed = 0

API_URL = "https://api.meta.ai/v1/chat/completions"
MODEL = "muse-spark-1.3"

def load_api_key():
    try:
        from google.colab import userdata
        k = (userdata.get("MODEL_API_KEY") or "").strip()
        if k: return k
    except: pass
    k = os.environ.get("MODEL_API_KEY","").strip()
    if k: return k
    try: return getpass("Paste Meta API Key (Required for 100+ Languages): ").strip()
    except: return ""

API_KEY = load_api_key()
print("ONLINE: 100+ Language AI ON" if API_KEY else "OFFLINE: Dictionary Mode ON")

LINGUA = None
try:
    from lingua import LanguageDetectorBuilder
    LINGUA = LanguageDetectorBuilder.from_all_languages().with_preloaded_language_models().build()
    print("Lingua Loaded")
except Exception as e:
    print("Lingua failed:", e)

SYSTEM_PROMPT = """You are world-best moderator for 100+ languages.
CRITICAL SARCASM RULE: Praise + Negative Result/Action = Sarcasm Toxic 70-80, NEVER Safe.
- "Oh la la, tu es vraiment un genie pour tout gacher encore une fois" = French, Sarcasm 76
- "Sei un genio assoluto, fai schifo" = Italian, Sarcasm 74
- "Great job, you ruined everything again" = English, Sarcasm 76
- "Parabens pelo excelente trabalho realizado" with NO negative = Safe 4
- "bu onye nzuzu kpamkpam" = Igbo, Toxic 78
RULES: Insult=70-84, Severe=85-92, Threat=93-99, Mild negative=48, NEVER 100.
Return ONLY JSON: {"status":"Safe|Warning|Toxic","toxicity_score":0-99,"category":"Safe|Hate Speech|Harassment|Cyberbullying|Sarcasm|Threat|Profanity|Spam|Self-harm","detected_language":"Language","country":"Country","language_type":"Family","translation":"English meaning","action":"Approve|Flag for Review|Report/Delete","explanation":"English"}"""

VERIFY_PROMPT = """Check for hidden sarcasm: Is praise used mockingly with a negative action/result? Example: genius + ruined everything = Sarcasm Toxic.
Return ONLY JSON: {"is_toxic":true/false,"language":"Language","country":"Country","type":"Sarcasm|Harassment|Profanity|Threat|Safe","score":0-99,"translation":"English","reason":"why"}"""

history_data, cache, session = [], {}, requests.Session()

LANG_INFO={
"Igbo":("Niger-Congo","Nigeria"),"Yoruba":("Niger-Congo","Nigeria"),"Hausa":("Afro-Asiatic","Nigeria / Niger"),
"Swahili":("Niger-Congo (Bantu)","Tanzania / Kenya"),"Bengali":("Indo-European (Indo-Aryan)","Bangladesh / India"),
"Italian":("Indo-European (Romance)","Italy"),"Spanish":("Indo-European (Romance)","Spain / Mexico"),
"French":("Indo-European (Romance)","France"),"Portuguese":("Indo-European (Romance)","Brazil / Portugal"),
"German":("Indo-European (Germanic)","Germany"),"Arabic":("Semitic","Saudi Arabia / Egypt / UAE"),
"Japanese":("Japonic","Japan"),"Chinese (Simplified)":("Sino-Tibetan","China"),"Chinese (Traditional)":("Sino-Tibetan","Taiwan / Hong Kong"),
"Korean":("Koreanic","South Korea"),"English":("Indo-European (Germanic)","USA / UK"),
"Hindi / Marathi":("Indo-European (Indo-Aryan)","India"),"Urdu":("Indo-European (Indo-Aryan)","Pakistan"),
"Russian / Ukrainian":("Indo-European (Slavic)","Russia / Ukraine"),"Tamil":("Dravidian","India"),
"Thai":("Tai-Kadai","Thailand"),"Greek":("Indo-European","Greece"),"Hebrew":("Semitic","Israel")
}

SCRIPTS=[(r'[\u3040-\u30FF]','Japanese'),(r'[\uAC00-\uD7AF]','Korean'),(r'[\u0980-\u09FF]','Bengali'),(r'[\u0900-\u097F]','Hindi / Marathi'),(r'[\u0B80-\u0BFF]','Tamil'),(r'[\u0E00-\u0E7F]','Thai'),(r'[\u0400-\u04FF]','Russian / Ukrainian'),(r'[\u0370-\u03FF]','Greek'),(r'[\u0590-\u05FF]','Hebrew')]
TRAD=set('們這裡愛國學麼為妳兒點對開關')

def detect_language(text):
    t=text.strip(); low=" "+t.lower()+" "
    for pat,name in SCRIPTS:
        if re.search(pat,t):
            fam,c=LANG_INFO.get(name,("World Language","Multiple")); return name,fam,c,1.0
    if re.search(r'[\u0600-\u06FF]',t):
        lang="Urdu" if any(c in t for c in 'ےٹڈڑ') else "Arabic"
        fam,c=LANG_INFO.get(lang,("Semitic","Multiple")); return lang,fam,c,1.0
    if re.search(r'[\u4E00-\u9FFF]',t):
        lang="Chinese (Traditional)" if any(c in TRAD for c in t) else "Chinese (Simplified)"
        return lang,LANG_INFO[lang][0],LANG_INFO[lang][1],1.0
    kw={
     "French":['tu es','vraiment','gâcher','gacher','oh la la','encore une fois','félicitations','excellent travail','je te déteste','stupide','connard'],
     "Portuguese":['parabéns','parabens','excelente trabalho','trabalho realizado','muito bem','obrigado','idiota','palhaço'],
     "Italian":['sei un','sei uno','fai schifo','schifo','grazie mille','complimenti','genio','cretino','pagliaccio'],
     "Spanish":['eres un','felicitaciones','excelente trabajo','muchas gracias','payaso','inútil','idiota'],
     "German":['du bist','herzlichen glückwunsch','großartige arbeit','trottel','wertlos'],
     "Igbo":['bụ','onye nzuzu','nzuzu','kpamkpam','onye'],
     "Swahili":['wewe ni','mjinga','mpumbavu','asante','hongera'],
     "Bengali":['মারা যাবি','মরে যাবি','মাথায় গোবর','গাধা','মাকাল ফল','অপদার্থ'],
     "Hindi / Marathi":['ullu ka pattha','nalayak','बेवकूफ','उल्लू का पट्ठा'],
    }
    for lang,words in kw.items():
        if any(w in low for w in words):
            fam,c=LANG_INFO.get(lang,("World Language","Multiple")); return lang,fam,c,0.98
    if LINGUA:
        try:
            l=LINGUA.detect_language_of(t)
            if l:
                name=str(l).split('.')[-1].replace('_',' ').title()
                fam,c=LANG_INFO.get(name,("World Language","Multiple")); return name,fam,c,0.90
        except: pass
    try:
        code=str(detect_langs(t)[0].lang)
        mp={'en':'English','fr':'French','pt':'Portuguese','it':'Italian','es':'Spanish','de':'German','ig':'Igbo','bn':'Bengali','hi':'Hindi / Marathi','ar':'Arabic','ur':'Urdu','sw':'Swahili','yo':'Yoruba','ha':'Hausa'}
        name=mp.get(code,code.upper()); fam,c=LANG_INFO.get(name,("World Language","Unknown")); return name,fam,c,0.70
    except: return "Unknown","Unknown","Unknown",0.0

# ======= UNIVERSAL SARCASM ENGINE =======
PRAISE_WORDS=['génie','genie','génial','genial','genius','brilliant','great job','good job','well done','bravo','bravissimo','genio','genio assoluto','excelente','parabéns','parabens','congratulations','félicitations','felicitaciones','complimenti','অভিনন্দন','দারুণ','বাহ','খুব ভালো কাজ','رائع','شاطر','すごい','天才','잘했어','बहुत बढ़िया','शाबाश']
NEGATIVE_ACTION=['gâcher','gacher','tout gâcher','ruiner','ruin','ruined everything','mess up','messed up','screw up','encore raté',"n'importe quoi",'catastrophe','fai schifo','schifo','you suck','failed again','সব নষ্ট','সব শেষ করে','আবার নষ্ট','সব গুবলেট','পণ্ড করে','todo estragado','arruinar','lo arruinaste','wieder ruiniert','alles ruiniert','台無し','台なし','また失敗','또 망쳤','सब बर्बाद','सब खराब कर']
POSITIVE_PHRASES=['parabéns pelo excelente trabalho','excelente trabalho realizado','congratulations','great job','excellent work','thank you','merci beaucoup pour','félicitations pour','excellent travail','grazie mille','muchas gracias','অভিনন্দন','ধন্যবাদ','খুব ভালো কাজ হয়েছে','শুভকামনা','شكرا لك','ありがとう','감사합니다','谢谢你','बहुत धन्यवाद','शुक्रिया']

# ======= TOXIC + সূক্ষ্ম গালি + প্রবাদ / স্ল্যাং =======
TOXIC_WORDS={
'onye nzuzu':78,'nzuzu':72,'idiot':75,'stupid':75,'stupide':75,'idiota':75,'stupido':75,'fai schifo':76,'schifo':70,
'বোকা':72,'গাধা':75,'খানকি':88,'傻逼':88,'白痴':75,'バカ':75,'馬鹿':75,'씨발':88,'병신':84,'غبي':75,'burro':72,'cretino':75,
# বাংলা সূক্ষ্ম
'মারা যাবি':93,'মরে যাবি':93,'মরে যা':90,'তুই মর':90,'মাথায় গোবর':70,'মাথায় গোবর':70,'গোবর গণেশ':70,'ছাগল':68,'পাগল ছাগল':72,'মাকাল ফল':68,'ধান্দাবাজ':70,'চামচা':66,'শুয়োরের বাচ্চা':84,'কুত্তার বাচ্চা':84,'হারামি':80,'হারামজাদা':84,'পাগল':70,'শুয়োর':78,'কুত্তা':76,'অপদার্থ':74,'অকর্মা':72,'কোনো কাজের না':74,
'mara jabi':93,'more jabi':93,'more ja':90,'mathay gobor':70,'chagol':68,'makal fol':68,'dhandabaz':70,'bokachoda':88,'magi':86,'khanki':88,
# Hindi/Urdu
'ullu ka pattha':78,'उल्लू का पट्ठा':78,'गधे का बच्चा':80,'नालायक':72,'चमचा':66,'चूतिया':88,'हरामी':80,'बेवकूफ':75,'पागल':70,'कुत्ता':76,'nalayak':72,'chamcha':66,'bewakoof':75,'chutiya':88,'احمق':75,'يا حمار':78,'يا كلب':80,'يا غبي':75,'ابن الكلب':84,'يا تافه':70,'بے وقوف':75,
# English subtle
'loser':72,'pathetic':70,'worthless':78,'you are a clown':70,'trash':72,'disgrace':72,'shame on you':66,'brainless':75,'moron':75,'dumb':72,'useless':74,'you suck':76,'hate you':74,
# French/Italian/Spanish/Portuguese/German
"espèce d'idiot":78,'imbécile':78,'connard':84,'merde':78,'je te déteste':74,'tu es nul':72,'bouffon':72,'sei un pagliaccio':72,'non vali niente':74,'eres un payaso':72,'eres un inútil':76,'no sirves para nada':74,'és um palhaço':72,'não vales nada':74,'du bist wertlos':78,'du trottel':72,
# African / Turkish / Russian / Indonesian
'oponu':75,'ode ni':75,'dan iska':80,'mjinga':75,'mpumbavu':76,'pumbavu':76,'aptal':75,'salak':75,'gerizekalı':78,'дурак':75,'идиот':75,'тупой':75,'сука':86,'bodoh':75,'tolol':78,'goblok':80
}

THREATS=['kill you','i will kill','you will die','you are going to die','মেরে ফেল','মেরে ফেলব','খুন করব','মারা যাবি','মরে যাবি','তুই মর','মরে যা','তোকে মারব','mara jabi','more jabi','ti uccido','te mato','je te tue','vas a morir','te vas a morir','سأقتلك','اقتلك','ستموت','راح تموت','杀了你','你会死','殺してやる','死ね','मार डालूंगा','मार दूंगा','तुम मर जाओगे','तू मर जा','jaan se maar']
MILD=['bad person','nobody likes you','no one likes you','কেউ পছন্দ করে না','তুমি খারাপ মানুষ','শخص سيء','شخص سيء','لا نريد رؤيتك','तुम बुरे हो','कोई पसंद नहीं करता']

SUBTLE_PATTERNS=[
(r'মাথায়|মাথায়.*গোবর|গোবর.*মাথা',70,"Bengali idiom insult: implying brainless"),
(r'(কোনো কাজের না|অকর্মা|অপদার্থ)',74,"Bengali subtle insult: worthless/useless person"),
(r'nobody (likes|loves) you|no one (likes|loves) you',72,"Social exclusion insult"),
(r'তোকে কেউ পছন্দ করে না|কেউ তোকে.*পছন্দ করে না',72,"Bengali social exclusion"),
(r"(you are|you're) (a )?(clown|loser|joke|trash)",72,"English subtle insult"),
(r'(eres|sei|tu es|és) (un |um |une )?(payaso|pagliaccio|idiot|palhaço|nul)',75,"Romance language subtle insult"),
(r'ullu ka pattha|ullu ke patthe',78,"Hindi idiom: fool"),
]

def find_sarcasm(t):
    praise=[w for w in PRAISE_WORDS if w in t]
    neg=[w for w in NEGATIVE_ACTION if w in t]
    return (praise,neg) if praise and neg else (None,None)

def offline(text,lang,fam,country):
    t=text.lower()
    base={"detected_language":lang,"language_type":fam,"country":country}
    if any(w in t for w in THREATS):
        return {**base,"status":"Toxic","toxicity_score":96,"category":"Threat","action":"Report/Delete","explanation":"Direct threat / death wish detected."}
    praise,neg=find_sarcasm(t)
    if praise:
        return {**base,"status":"Toxic","toxicity_score":76,"category":"Sarcasm","action":"Flag for Review","explanation":f"Sarcasm detected: mocking praise '{praise[0]}' combined with negative action '{neg[0]}'. Praise is used to insult, not to appreciate."}
    for pat,sc,exp in SUBTLE_PATTERNS:
        if re.search(pat,t):
            return {**base,"status":"Toxic","toxicity_score":sc,"category":"Harassment","action":"Flag for Review","explanation":f"Subtle / idiomatic insult detected: {exp}"}
    found=[(w,s) for w,s in TOXIC_WORDS.items() if w in t]
    if found:
        w,s=max(found,key=lambda x:x[1])
        cat="Profanity" if s>=85 else "Harassment"
        act="Report/Delete" if s>=85 else "Flag for Review"
        return {**base,"status":"Toxic","toxicity_score":s,"category":cat,"action":act,"explanation":f"Toxic word/slang '{w}' detected in {lang}."}
    if any(w in t for w in MILD):
        return {**base,"status":"Warning","toxicity_score":48,"category":"Harassment","action":"Flag for Review","explanation":"Mild negative language, no profanity."}
    if any(w in t for w in POSITIVE_PHRASES):
        return {**base,"status":"Safe","toxicity_score":4,"category":"Safe","action":"Approve","explanation":f"Positive / congratulatory content in {lang}. No insult, sarcasm or threat found."}
    if lang=="Unknown":
        return {**base,"status":"Warning","toxicity_score":40,"category":"Cyberbullying","action":"Flag for Review","explanation":"Language not identified offline. API Key required."}
    return {**base,"status":"Safe","toxicity_score":5,"category":"Safe","action":"Approve","explanation":"No toxic, sarcastic or negative content found."}

def api_call(system,user):
    try:
        r=session.post(API_URL,headers={"Authorization":f"Bearer {API_KEY}","Content-Type":"application/json"},json={"model":MODEL,"messages":[{"role":"system","content":system},{"role":"user","content":user}],"temperature":0.0,"max_tokens":400},timeout=(3,25))
        if r.status_code!=200: return None
        m=re.search(r'\{.*\}',r.json()["choices"][0]["message"]["content"],re.DOTALL)
        return json.loads(m.group()) if m else None
    except: return None

def level(s):
    return "Level 0 - Safe (0-29)" if s<30 else "Level 1 - Warning (30-59)" if s<60 else "Level 2 - Toxic (60-84)" if s<85 else "Level 3 - Severe (85-99)"

def analyze(text):
    empty=pd.DataFrame(history_data,columns=["Time","Comment","Language","Country","Level","Status","Score","Category","Action"])
    if not text or len(text.strip())<2:
        return "<h3>Please enter a comment first</h3>","","","","","",empty,None
    start=time.time(); lang,fam,country,conf=detect_language(text); key=text.strip().lower()
    if key in cache:
        data,elapsed=cache[key],0.01
    else:
        data=offline(text,lang,fam,country)
        off_cat,off_score=data["category"],int(data["toxicity_score"])
        if API_KEY and off_cat not in ["Threat","Sarcasm"]:
            main=api_call(SYSTEM_PROMPT,f"Local Detector: {lang} ({country})\nText: {text}")
            if main:
                try:
                    if int(main.get("toxicity_score",0)) > off_score: data=main
                except: pass
            if int(data.get("toxicity_score",0))<65 and off_cat=="Safe":
                v=api_call(VERIFY_PROMPT,f"Text: {text}")
                if v and v.get("is_toxic"):
                    data={"status":"Toxic","toxicity_score":max(70,int(v.get("score",76))),"category":v.get("type","Sarcasm"),"detected_language":v.get("language",lang),"country":v.get("country",country),"language_type":fam,"action":"Flag for Review","explanation":f"Second Check: {v.get('translation','')} - {v.get('reason','')}"}
        if off_cat in ["Sarcasm","Threat"] or off_score>=70:
            data=offline(text,lang,fam,country)
        elif off_cat=="Safe" and off_score==4:
            t=text.lower()
            if not find_sarcasm(t)[0] and not [w for w in TOXIC_WORDS if w in t]:
                data.update({"status":"Safe","toxicity_score":4,"category":"Safe","action":"Approve"})
        try: sc=int(data.get("toxicity_score",5))
        except: sc=5
        sc=max(0,min(99,sc)); status="Safe" if sc<30 else "Warning" if sc<60 else "Toxic"
        final_lang=lang if conf>=0.90 else data.get("detected_language",lang)
        fam2,country2=LANG_INFO.get(final_lang,(data.get("language_type",fam),data.get("country",country)))
        data.update({"toxicity_score":sc,"status":status,"detected_language":final_lang,"country":country2,"language_type":fam2,"action":"Approve" if status=="Safe" else "Flag for Review" if status=="Warning" else "Report/Delete"})
        lang,fam,country=final_lang,fam2,country2
        elapsed=round(time.time()-start,2); cache[key]=data
    score,status,lv=int(data["toxicity_score"]),data["status"],level(int(data["toxicity_score"]))
    color,bg,icon=("#28a745","#e8f5e9","SAFE") if score<30 else ("#e6a817","#fffde7","WARNING") if score<60 else ("#dc3545","#ffebee","TOXIC")
    report=f"""<div style="border:3px solid {color};border-radius:12px;padding:20px;text-align:center;background:{bg};font-family:Arial;"><h2 style="color:{color};margin:0;">{icon}</h2><div style="font-size:42px;font-weight:bold;">{score}%</div><div>{lv} | {elapsed}s</div><p>"{html.escape(text[:80])}"</p><div style="background:#e0e0e0;height:22px;border-radius:6px;"><div style="background:{color};width:{score}%;height:100%;border-radius:6px;"></div></div></div>"""
    history_data.insert(0,{"Time":datetime.now().strftime("%H:%M:%S"),"Comment":text[:60],"Language":lang,"Country":country,"Level":lv,"Status":status,"Score":f"{score}%","Category":data.get("category"),"Action":data.get("action")})
    history_data[:]=history_data[:50]
    tmp=tempfile.NamedTemporaryFile(mode='w',suffix='.txt',delete=False,encoding='utf-8')
    tmp.write(f"Text: {text}\nLanguage: {lang}\nCountry: {country}\nFamily: {fam}\nLevel: {lv}\nScore: {score}%\nCategory: {data.get('category')}\nExplanation: {data.get('explanation')}")
    tmp.close()
    return report,f"{lang} | Family: {fam}",country,lv,data.get("category"),data.get("action"),data.get("explanation"),pd.DataFrame(history_data),tmp.name

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"),title="Cyberbullying Shield") as demo:
    gr.Markdown("# Cyberbullying Shield - 100+ Languages FINAL")
    gr.Markdown("### Multilingual AI Moderation System for Cyberbullying, Hate Speech, Sarcasm and Threat Detection in 100+ Languages")
    with gr.Tab("Live Demo"):
        inp=gr.Textbox(label="Comment in ANY Language",lines=3,placeholder="Type a comment in any language of the world...")
        btn=gr.Button("Analyze Comment",variant="primary",size="lg")
        with gr.Row():
            with gr.Column(scale=1): out_html=gr.HTML()
            with gr.Column(scale=2):
                lang_o=gr.Textbox(label="Detected Language and Family"); country_o=gr.Textbox(label="Country"); level_o=gr.Textbox(label="Toxicity Level"); cat_o=gr.Textbox(label="Category"); act_o=gr.Textbox(label="Recommended Action"); exp_o=gr.Textbox(label="Explanation in English",lines=4)
        file_o=gr.File(label="Download Report")
        gr.Examples([
            ["Oh la la, tu es vraiment un génie pour tout gâcher encore une fois."],
            ["Félicitations pour ton excellent travail, merci beaucoup"],
            ["Parabéns pelo excelente trabalho realizado"],
            ["Sei un genio assoluto, fai schifo."],
            ["Great job, you ruined everything again"],
            ["Congratulations, excellent work my friend"],
            ["bụ onye nzuzu kpamkpam."],
            ["You are an idiot, nobody likes you"],
            ["তুই তো মারা যাবি"],
            ["তোর মাথায় গোবর আছে, তুই একটা মাকাল ফল"],
            ["তুই একটা গাধা, সব নষ্ট করে দিলি"],
            ["বাহ দারুণ কাজ করেছিস, সব শেষ করে দিলি"],
            ["অভিনন্দন, খুব ভালো কাজ হয়েছে"],
            ["ullu ka pattha, tum ek bewakoof ho"],
            ["You are a clown, nobody likes you"]
        ],inputs=inp,label="Examples - Click to Test")
    with gr.Tab("History"): hist=gr.Dataframe(label="Moderation History")
    btn.click(analyze,inputs=inp,outputs=[out_html,lang_o,country_o,level_o,cat_o,act_o,exp_o,hist,file_o])

demo.queue(); demo.launch(share=True,debug=False)

Paste Meta API Key (Required for 100+ Languages): ··········
ONLINE: 100+ Language AI ON
Lingua Loaded


/tmp/ipykernel_777/1033319053.py:224: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"),title="Cyberbullying Shield") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://874b20bb065893dd51.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
